# Lab 04.5 — Contours and Object Tracking

## Objectives

- detect the largest object of a selected color;
- compute its contour area;
- draw a bounding box;
- compute its center;
- classify the object's horizontal position as LEFT, CENTER or RIGHT.

This is the first simple tracking experiment.

## Camera setup — provided code

Run this helper cell once in this notebook. It uses the previously verified **V4L2 / `/dev/video0` → 20 warm-up frames → capture → inline JPEG display → release** sequence. The requested resolution is 640×480; use `frame.shape` to check the actual resolution.

The `with camera_session()` block releases the camera through `finally`, including on Python errors or a normal Jupyter interrupt. Only one notebook/kernel should use the camera at a time. If Linux assigns a different capture device, change **CAMERA_DEVICE in each notebook you use**; notebook variables are not shared.

The helper is supplied; you do not need to implement it. Time limits are checked between reads: a USB driver call that blocks can delay an interrupt or timeout.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
WARMUP_FRAMES = 20

def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and close other camera notebooks."
        )
    return frame

@contextmanager
def camera_session():
    """Provided helper: open, warm up, and always release the camera."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError(
                "Cannot open " + CAMERA_DEVICE +
                ". Check the device and shut down other kernels using the camera."
            )
        # Request a modest resolution; the driver may negotiate another one.
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        warmup_start = time.monotonic()
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)  # Discard frames while exposure settles.
            if time.monotonic() - warmup_start > 10:
                raise RuntimeError("Camera warm-up is too slow. Check the USB connection.")
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))


In [ ]:
TARGET = "GREEN"

RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (34, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

In [ ]:
def color_mask(hsv, color_name):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in RANGES[color_name]:
        mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def horizontal_region(cx, width):
    if cx < width / 3:
        return "LEFT"
    elif cx < 2 * width / 3:
        return "CENTER"
    else:
        return "RIGHT"

## What the measurements mean

A contour follows the boundary of a white region in the mask. `contourArea()` estimates the area enclosed by that boundary; it is not exactly the count of white pixels. We reject small contours, then select the largest remaining one.

`boundingRect()` gives `x, y, w, h`: the upper-left corner, width and height of an enclosing rectangle. The **centroid** is the region's center of area:

`cx = M["m10"] / M["m00"]`, `cy = M["m01"] / M["m00"]`.

The `m00 != 0` check avoids division by zero. The centroid need not equal the rectangle's center. Image coordinates start at the top left: x grows rightward and y grows downward.

LEFT / CENTER / RIGHT refer to thirds of the displayed image, which is not mirrored. This simple tracker selects the largest matching object in each frame; it may switch targets when two same-colored objects are present.

## Run the tracker

Hold a large, saturated colored object in front of the camera. The session stops after at most 180 processed frames or approximately 15 seconds after warm-up. These are limits checked between reads, not a guaranteed driver timeout. Use **Kernel → Interrupt** to stop early; rerun this cell for another session.

640×480 is requested. If you use another resolution, the fixed `MIN_AREA` threshold may need adjustment. `time.sleep(0.03)` is only a short pause; it does not guarantee a frame rate. Display and processing also take time.

In [ ]:
MIN_AREA = 1200  # Contour area threshold; pixels squared.
MAX_FRAMES = 180
MAX_SECONDS = 15

with camera_session() as cap:
    start = time.monotonic()
    for _ in range(MAX_FRAMES):
        if time.monotonic() - start >= MAX_SECONDS:
            break
        frame = read_frame(cap)

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = color_mask(hsv, TARGET)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        valid = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]

        label = TARGET + ": LOST"
        if valid:
            cnt = max(valid, key=cv2.contourArea)
            area = cv2.contourArea(cnt)
            x, y, w, h = cv2.boundingRect(cnt)

            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])

                region = horizontal_region(cx, frame.shape[1])

                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 6, (255, 255, 255), -1)

                label = f"{TARGET}: {region}  center=({cx},{cy}) area={int(area)}"

        cv2.putText(frame, label, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255, 255, 255), 2)

        show_bgr(frame, replace=True)

        time.sleep(0.03)



## Short exercise

1. Move a green object through all three regions of the image.
2. Remove it and check the **LOST** label.
3. Observe how the reported area changes as you move the same object closer.

**Record:** one centroid coordinate and its LEFT/CENTER/RIGHT region. No additional tracking algorithm is required.